1.1. Реалізувати процедуру завантаження файлів з VHI-індексом для кожної адміністративної одиниці України (urllib). 
Додати мітку часу до імені файлу та реалізувати механізм запобігання повторному завантаженню.

In [23]:
import urllib.request
import os
from datetime import datetime

def download_vhi_data(directory='vhi_data'):
    # Створюємо папку, якщо вона не існує
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Директорію '{directory}' створено.")

    # Базовий URL для України
    base_url = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={}&year1=1981&year2=2024&type=Mean"

    for i in range(1, 28): # ID 1-27 (ID 0 ігноруємо за завданням)
        # Перевіряємо, чи ми вже завантажували дані для цієї області сьогодні
        existing_files = [f for f in os.listdir(directory) if f.startswith(f"vhi_id_{i}_")]
        
        if existing_files:
            print(f"Область {i}: файл уже існує ({existing_files[0]}). Пропускаємо.")
            continue

        url = base_url.format(i)
        now = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"vhi_id_{i}_{now}.csv"
        path = os.path.join(directory, filename)

        try:
            print(f"Завантаження області {i}...", end=" ")
            urllib.request.urlretrieve(url, path)
            print("Готово.")
        except Exception as e:
            print(f"Помилка: {e}")

# Виклик процедури
download_vhi_data()

Область 1: файл уже існує (vhi_id_1_20260417_122259.csv). Пропускаємо.
Область 2: файл уже існує (vhi_id_2_20260417_122303.csv). Пропускаємо.
Область 3: файл уже існує (vhi_id_3_20260417_122304.csv). Пропускаємо.
Область 4: файл уже існує (vhi_id_4_20260417_122307.csv). Пропускаємо.
Область 5: файл уже існує (vhi_id_5_20260417_122308.csv). Пропускаємо.
Область 6: файл уже існує (vhi_id_6_20260417_122309.csv). Пропускаємо.
Область 7: файл уже існує (vhi_id_7_20260417_122310.csv). Пропускаємо.
Область 8: файл уже існує (vhi_id_8_20260417_122311.csv). Пропускаємо.
Область 9: файл уже існує (vhi_id_9_20260417_122311.csv). Пропускаємо.
Область 10: файл уже існує (vhi_id_10_20260417_122312.csv). Пропускаємо.
Область 11: файл уже існує (vhi_id_11_20260417_122313.csv). Пропускаємо.
Область 12: файл уже існує (vhi_id_12_20260417_122314.csv). Пропускаємо.
Область 13: файл уже існує (vhi_id_13_20260417_122315.csv). Пропускаємо.
Область 14: файл уже існує (vhi_id_14_20260417_122316.csv). Пропускає

1.2. Зчитати завантажені файли у pandas dataframe. Здійснити data cleaning (прибрати зайвий текст, заповнити пропуски). 
Реалізувати процедуру зміни індексів (NOAA English -> Ukrainian Alphabet).

In [24]:
import pandas as pd
import glob
import os
import re

def create_cleaned_dataframe(directory='vhi_data'):
    all_files = glob.glob(os.path.join(directory, "*.csv"))
    list_df = []
    
    reindex_map = {
        24: 1, 25: 2, 5: 3, 6: 4, 27: 5, 23: 6, 26: 7, 7: 8, 11: 9, 13: 10, 
        14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16, 21: 17, 22: 18, 
        8: 19, 9: 20, 10: 21, 1: 22, 3: 23, 2: 24, 4: 25, 12: 26, 20: 27
    }

    for file in all_files:
        filename = os.path.basename(file)
        match = re.search(r'id_(\d+)_', filename)
        if not match: continue
        old_id = int(match.group(1))
        new_id = reindex_map.get(old_id, old_id)

        # 1. Читаємо файл, забороняючи робити будь-яку колонку індексом
        df_temp = pd.read_csv(file, header=1, sep=',', index_col=False, engine='python')
        
        # 2. Вичищаємо теги з назв колонок та самих даних
        df_temp.columns = [re.sub(r'<[^>]*>', '', col).strip() for col in df_temp.columns]
        df_temp = df_temp.astype(str).replace(r'<[^>]*>', '', regex=True).apply(lambda x: x.str.strip())
        
        # 3. Примусово беремо перші 7 колонок і даємо їм правильні імена
        # Це лікує ситуацію, коли через зайву кому в кінці рядка все зсувається
        df_temp = df_temp.iloc[:, :7]
        df_temp.columns = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']
        
        # 4. Перетворюємо на числа
        df_temp = df_temp.apply(pd.to_numeric, errors='coerce')
        
        # 5. Видаляємо сміттєві рядки (де не розпізнався рік або VHI)
        df_temp = df_temp.dropna(subset=['Year', 'VHI'])

        if not df_temp.empty:
            df_temp['Area_ID'] = new_id
            list_df.append(df_temp)

    if not list_df:
        return pd.DataFrame()
        
    return pd.concat(list_df, ignore_index=True)

# ПЕРЕЗАПУСК ТА ПЕРЕВІРКА
df = create_cleaned_dataframe()
print(f"Результат: {len(df)} рядків.")
if not df.empty:
    display(df.head())

Результат: 60372 рядків.


,Year,Week,SMN,SMT,VCI,TCI,VHI,Area_ID
0,1982.0,1.0,0.059,258.24,51.11,48.78,49.95,21
1,1982.0,2.0,0.063,261.53,55.89,38.20,47.04,21
2,1982.0,3.0,0.063,263.45,57.30,32.69,44.99,21
3,1982.0,4.0,0.061,265.10,53.96,28.62,41.29,21
4,1982.0,5.0,0.058,266.42,46.87,28.57,37.72,21


1.3. Реалізувати функцію для отримання ряду VHI для області за вказаний рік.

In [25]:
def get_vhi_series(df, province_id, year):
    result = df[(df['Area_ID'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]
    return result

print("VHI для області 1 (Вінницька) за 2020 рік:")
display(get_vhi_series(df, province_id=1, year=2020).head())

VHI для області 1 (Вінницька) за 2020 рік:


,Week,VHI
35516,1.0,40.92
35517,2.0,43.19
35518,3.0,44.74
35519,4.0,45.29
35520,5.0,44.80


1.4. Реалізувати процедуру для формування вибірки: ряд VHI за вказаний діапазон років для вказаних областей.

In [26]:
def get_vhi_range(df, provinces, start_year, end_year):
    # provinces має бути списком, наприклад [1, 5, 10]
    result = df[(df['Area_ID'].isin(provinces)) & 
                (df['Year'] >= start_year) & 
                (df['Year'] <= end_year)]
    return result

print("VHI для областей 1 та 10 за 2015-2017 роки:")
display(get_vhi_range(df, provinces=[1, 10], start_year=2015, end_year=2017).head(10))

VHI для областей 1 та 10 за 2015-2017 роки:


,Year,Week,SMN,SMT,VCI,TCI,VHI,Area_ID
8424,2015.0,1.0,0.044,257.32,36.02,53.71,44.84,10
8425,2015.0,2.0,0.042,255.91,36.46,56.68,46.54,10
8426,2015.0,3.0,0.041,255.79,35.55,56.03,45.78,10
8427,2015.0,4.0,0.044,257.36,36.81,51.01,43.91,10
8428,2015.0,5.0,0.049,260.20,38.72,44.00,41.36,10
8429,2015.0,6.0,0.056,262.72,40.83,40.94,40.89,10
8430,2015.0,7.0,0.066,265.49,43.31,38.94,41.12,10
8431,2015.0,8.0,0.081,270.65,47.37,30.57,38.97,10
8432,2015.0,9.0,0.096,275.70,50.57,23.39,36.98,10
8433,2015.0,10.0,0.109,280.01,51.62,18.59,35.11,10


1.5. Реалізувати процедуру пошуку екстремумів (min та max), середнього та медіани для вказаних областей та років.

In [27]:
def get_vhi_statistics(df, province_id, year):
    filtered_df = df[(df['Area_ID'] == province_id) & (df['Year'] == year)]
    
    # Обчислюємо статистику для колонки VHI
    stats = filtered_df['VHI'].agg(['min', 'max', 'mean', 'median'])
    return stats

print(f"Статистика VHI для області 1 за 2020 рік:")
print(get_vhi_statistics(df, province_id=1, year=2020))

Статистика VHI для області 1 за 2020 рік:
min       34.480000
max       64.120000
mean      45.911538
median    44.230000
Name: VHI, dtype: float64


In [28]:
df['Area_ID'].unique()

array([21,  9, 26, 10, 11, 12, 13, 14, 15, 16, 22, 27, 17, 18,  6,  1,  2,
        7,  5, 24, 23, 25,  3,  4,  8, 19, 20])

In [29]:
import glob
import os

directory = 'vhi_data'
all_files = glob.glob(os.path.join(directory, "*.csv"))
print(f"Знайдено файлів у папці: {len(all_files)}")

if len(all_files) > 0:
    print(f"Приклад шляху до файлу: {all_files[0]}")
    

Знайдено файлів у папці: 27
Приклад шляху до файлу: vhi_data\vhi_id_10_20260417_122312.csv


In [30]:
import pandas as pd
import glob
import os

# Беремо перший ліпший файл для тесту
test_file = glob.glob(os.path.join('vhi_data', "*.csv"))[0]

# Читаємо БЕЗ фільтрів, щоб побачити сирі дані
test_df = pd.read_csv(test_file, header=1)
print("Назви колонок, які бачить Pandas:")
print(test_df.columns.tolist())
print("\nПерші 3 рядки:")
print(test_df.head(3))

Назви колонок, які бачить Pandas:
['year', 'week', ' SMN', 'SMT', 'VCI', 'TCI', ' VHI<br>']

Перші 3 рядки:
               year   week     SMN    SMT    VCI    TCI   VHI<br>
<tt><pre>1982   1.0  0.059  258.24  51.11  48.78  49.95       NaN
1982            2.0  0.063  261.53  55.89  38.20  47.04       NaN
1982            3.0  0.063  263.45  57.30  32.69  44.99       NaN
